## FIAP Connect — IA de Compatibilidade de Grupos
### Alexis Rondo (RM 560384) | Vinicius Rodrigues (RM 559611) | 2TDSPS

In [1]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Criação do dataframe com dados simulados do FIAP Connect
# Cada registro representa uma combinação aluno + grupo
np.random.seed(42)

registros = []

for _ in range(300):
    num_hab_aluno = np.random.randint(1, 8)
    num_faltantes = np.random.randint(1, 8)
    num_cobertas = np.random.randint(0, min(num_hab_aluno, num_faltantes) + 1)
    percentual = round((num_cobertas / num_faltantes) * 100, 1) if num_faltantes > 0 else 0
    mesmo_periodo = np.random.choice([0, 1], p=[0.3, 0.7])
    mesma_unidade = np.random.choice([0, 1], p=[0.4, 0.6])
    vagas = np.random.choice([1, 2])

    # Regras baseadas na lógica real do FIAP Connect
    if num_cobertas == num_faltantes and mesmo_periodo == 1:
        compat = 'ALTA'
    elif percentual >= 50 and mesmo_periodo == 1:
        compat = 'MEDIA'
    elif percentual >= 30 and mesma_unidade == 1:
        compat = 'MEDIA'
    else:
        compat = 'BAIXA'

    # Ruído pra simular incerteza real
    if np.random.random() < 0.08:
        opcoes = ['ALTA', 'MEDIA', 'BAIXA']
        opcoes.remove(compat)
        compat = np.random.choice(opcoes)

    registros.append([num_hab_aluno, num_faltantes, num_cobertas,
                      percentual, mesmo_periodo, mesma_unidade, vagas, compat])

colunas = ['num_habilidades_aluno', 'num_habilidades_faltantes_grupo',
           'num_cobertas', 'percentual_cobertura',
           'mesmo_periodo', 'mesma_unidade', 'vagas_disponiveis', 'compatibilidade']

df = pd.DataFrame(registros, columns=colunas)

In [3]:
# Tamanho do dataframe
df.shape

(300, 8)

In [4]:
# Primeiros registros
df.head(10)

,num_habilidades_aluno,num_habilidades_faltantes_grupo,num_cobertas,percentual_cobertura,mesmo_periodo,mesma_unidade,vagas_disponiveis,compatibilidade
0,7,4,4,100.0,0,1,1,MEDIA
1,3,7,2,28.6,1,0,2,BAIXA
2,6,5,1,20.0,1,1,2,BAIXA
3,5,1,1,100.0,1,0,1,ALTA
4,3,7,1,14.3,0,1,1,BAIXA
5,7,6,2,33.3,1,1,1,ALTA
6,7,5,0,0.0,0,1,2,BAIXA
7,2,1,1,100.0,0,1,1,MEDIA
8,4,5,2,40.0,1,0,2,BAIXA
9,6,6,5,83.3,1,0,1,MEDIA


In [5]:
# Informações dos atributos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   num_habilidades_aluno            300 non-null    int64  
 1   num_habilidades_faltantes_grupo  300 non-null    int64  
 2   num_cobertas                     300 non-null    int64  
 3   percentual_cobertura             300 non-null    float64
 4   mesmo_periodo                    300 non-null    int64  
 5   mesma_unidade                    300 non-null    int64  
 6   vagas_disponiveis                300 non-null    int64  
 7   compatibilidade                  300 non-null    object 
dtypes: float64(1), int64(6), object(1)
memory usage: 18.9+ KB


In [6]:
# Estatísticas descritivas
df.describe()

,num_habilidades_aluno,num_habilidades_faltantes_grupo,num_cobertas,percentual_cobertura,mesmo_periodo,mesma_unidade,vagas_disponiveis
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,3.936667,3.913333,1.353333,38.737333,0.696667,0.636667,1.470000
std,2.042849,2.046080,1.342073,36.602836,0.460466,0.481763,0.499933
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,4.000000,4.000000,1.000000,28.600000,1.000000,1.000000,1.000000
75%,6.000000,6.000000,2.000000,66.700000,1.000000,1.000000,2.000000
max,7.000000,7.000000,6.000000,100.000000,1.000000,1.000000,2.000000


In [7]:
# Distribuição da variável alvo
df['compatibilidade'].value_counts()

,count
compatibilidade,
BAIXA,173
MEDIA,84
ALTA,43


In [8]:
# Conversão de dados categóricos em numéricos
le = LabelEncoder()
df['compatibilidade_cod'] = le.fit_transform(df['compatibilidade'])

# Mapeamento
for i, classe in enumerate(le.classes_):
    print(f'{classe} -> {i}')

ALTA -> 0
BAIXA -> 1
MEDIA -> 2


In [9]:
# Visualização após transformação
df[['compatibilidade', 'compatibilidade_cod']].head()

,compatibilidade,compatibilidade_cod
0,MEDIA,2
1,BAIXA,1
2,BAIXA,1
3,ALTA,0
4,BAIXA,1


In [10]:
# Variável X (features)
X = df[['num_habilidades_aluno', 'num_habilidades_faltantes_grupo',
        'num_cobertas', 'percentual_cobertura',
        'mesmo_periodo', 'mesma_unidade', 'vagas_disponiveis']]
X.head()

,num_habilidades_aluno,num_habilidades_faltantes_grupo,num_cobertas,percentual_cobertura,mesmo_periodo,mesma_unidade,vagas_disponiveis
0,7,4,4,100.0,0,1,1
1,3,7,2,28.6,1,0,2
2,6,5,1,20.0,1,1,2
3,5,1,1,100.0,1,0,1
4,3,7,1,14.3,0,1,1


In [11]:
# Variável y (alvo)
y = df['compatibilidade_cod'].values
y

array([2, 1, 1, 0, 1, 0, 1, 2, 1, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 2, 1, 2, 1, 1, 0, 1, 2, 0, 0, 2,
       1, 2, 1, 2, 1, 0, 0, 1, 0, 0, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 2, 1, 2, 1, 1, 0, 1, 2, 0, 0, 2, 2, 1, 0, 2, 2,
       1, 1, 0, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 2, 2, 1, 2, 2, 1, 1, 2, 1,
       1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1,
       2, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 1, 0, 1, 0, 1,
       1, 1, 1, 2, 0, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1,
       2, 1, 1, 1, 1, 2, 1, 1, 2, 1, 2, 2, 1, 0, 1, 0, 1, 1, 0, 2, 1, 1,
       0, 1, 0, 0, 1, 1, 1, 1, 2, 1, 1, 2, 0, 1, 2, 1, 0, 1, 2, 0, 2, 0,
       1, 2, 1, 1, 2, 2, 1, 2, 1, 0, 2, 1, 2, 2, 2, 0, 0, 1, 2, 1, 1, 1,
       2, 1, 1, 2, 2, 2, 2, 1, 2, 1, 2, 2, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0,
       0, 2, 1, 0, 0, 0, 1, 1, 1, 1, 2, 1, 2, 2, 1, 2, 1, 1, 1, 1, 2, 1,
       1, 2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 0, 1, 1])

In [12]:
# Separação 70% treino / 30% teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f'Treino: {X_train.shape[0]}')
print(f'Teste: {X_test.shape[0]}')

Treino: 210
Teste: 90


In [13]:
# Modelo 1 - Random Forest
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [14]:
# Predições do Random Forest
y_predict_rf = modelo_rf.predict(X_test)
y_predict_rf

array([1, 1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 2, 1, 2, 1, 1, 2, 1, 1, 1,
       1, 1, 2, 0, 1, 1, 2, 1, 1, 1, 2, 1, 1, 0, 2, 1, 2, 1, 1, 1, 1, 1,
       0, 1, 2, 2, 1, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1,
       1, 1, 1, 2, 2, 2, 1, 1, 2, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 2,
       1, 2])

In [15]:
# Acurácia do Random Forest
acc_rf = accuracy_score(y_test, y_predict_rf)
print(f'Acurácia: {acc_rf*100:.2f}%')

Acurácia: 85.56%


In [16]:
# Relatório do Random Forest
print(classification_report(y_test, y_predict_rf, target_names=le.classes_))

              precision    recall  f1-score   support

        ALTA       0.67      0.29      0.40         7
       BAIXA       0.88      0.96      0.92        54
       MEDIA       0.82      0.79      0.81        29

    accuracy                           0.86        90
   macro avg       0.79      0.68      0.71        90
weighted avg       0.85      0.86      0.84        90



In [17]:
# Modelo 2 - KNN
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_train, y_train)

KNeighborsClassifier()

In [18]:
# Predições do KNN
y_predict_knn = modelo_knn.predict(X_test)
y_predict_knn

array([0, 1, 0, 2, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 2, 2, 1, 1, 2, 1, 1,
       1, 2, 2, 0, 1, 1, 1, 2, 1, 1, 2, 1, 1, 0, 0, 1, 2, 1, 1, 1, 1, 1,
       0, 1, 2, 2, 1, 1, 1, 2, 2, 1, 2, 1, 1, 0, 1, 0, 2, 1, 1, 1, 1, 1,
       1, 1, 1, 2, 2, 2, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 0, 1, 2,
       1, 2])

In [19]:
# Acurácia do KNN
acc_knn = accuracy_score(y_test, y_predict_knn)
print(f'Acurácia: {acc_knn*100:.2f}%')

Acurácia: 75.56%


In [20]:
# Relatório do KNN
print(classification_report(y_test, y_predict_knn, target_names=le.classes_))

              precision    recall  f1-score   support

        ALTA       0.33      0.43      0.38         7
       BAIXA       0.84      0.85      0.84        54
       MEDIA       0.73      0.66      0.69        29

    accuracy                           0.76        90
   macro avg       0.63      0.65      0.64        90
weighted avg       0.76      0.76      0.76        90



In [21]:
# Comparação dos modelos
print(f'Random Forest: {acc_rf*100:.2f}%')
print(f'KNN:           {acc_knn*100:.2f}%')

melhor = 'Random Forest' if acc_rf >= acc_knn else 'KNN'
print(f'Melhor modelo: {melhor}')

Random Forest: 85.56%
KNN:           75.56%
Melhor modelo: Random Forest


In [22]:
# Teste com dados reais do Oracle APEX
# Beatriz (RM333333) busca grupo - ela domina IOT e SQA

dados_apex = pd.DataFrame([
    # Beatriz vs Grupo 1 (FIAP Connect Team) - faltam IOT e SQA, ela cobre as 2
    {'num_habilidades_aluno': 2, 'num_habilidades_faltantes_grupo': 2,
     'num_cobertas': 2, 'percentual_cobertura': 100.0,
     'mesmo_periodo': 1, 'mesma_unidade': 1, 'vagas_disponiveis': 1},

    # Beatriz vs Grupo 2 (Oracle Innovators) - faltam 5, ela cobre 2
    {'num_habilidades_aluno': 2, 'num_habilidades_faltantes_grupo': 5,
     'num_cobertas': 2, 'percentual_cobertura': 40.0,
     'mesmo_periodo': 1, 'mesma_unidade': 1, 'vagas_disponiveis': 2}
])

melhor_modelo = modelo_rf if acc_rf >= acc_knn else modelo_knn
predicoes = melhor_modelo.predict(dados_apex)
labels = le.inverse_transform(predicoes)

print(f'Beatriz -> Grupo 1 (FIAP Connect Team):  {labels[0]}')
print(f'Beatriz -> Grupo 2 (Oracle Innovators):   {labels[1]}')

Beatriz -> Grupo 1 (FIAP Connect Team):  ALTA
Beatriz -> Grupo 2 (Oracle Innovators):   MEDIA


In [23]:
# Salvando o modelo pra usar na Sprint 4 (API Flask)
with open('modelo_compatibilidade.pkl', 'wb') as f:
    pickle.dump(melhor_modelo, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('Modelo salvo: modelo_compatibilidade.pkl')
print('Encoder salvo: label_encoder.pkl')

Modelo salvo: modelo_compatibilidade.pkl
Encoder salvo: label_encoder.pkl
